In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_key = os.getenv("GROQ_API_KEY")

if groq_key:
    print(f"Groq API key loaded: {groq_key[:8]}...{groq_key[-4:]}")
else:
    print(f"Groq API key NOT found — check your .env file")


Groq API key loaded: gsk_jKpd...Kvja


In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("data/aws_unit_2.pdf")
pages = loader.load()

print(f"✅ Total pages loaded: {len(pages)}")
print(f"\n--- Page 1 Preview ---")
print(pages[0].page_content[:500])
print(f"\n--- Page 1 Metadata ---")
print(pages[0].metadata)

✅ Total pages loaded: 97

--- Page 1 Preview ---
AWS Solution Architect
(UNIT-2:Compute & Storage)
Dr. Shagun Sharma
Assistant Professor
School of Computing Science and Engineering
shagunsharma@vitbhopal.ac.in
1

--- Page 1 Metadata ---
{'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2026-03-15T04:19:45+00:00', 'moddate': '2026-03-15T04:19:46+00:00', 'source': 'data/aws_unit_2.pdf', 'total_pages': 97, 'page': 0, 'page_label': '1'}


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 20
)

chunks = splitter.split_documents(pages)

print(f"✅ Total chunks created: {len(chunks)}")
print(f"📄 Total pages: {len(pages)}")
print(f"\n--- Chunk 1 Content ---")
print(chunks[0].page_content)
print(f"\n--- Chunk 1 Metadata ---")
print(chunks[0].metadata)

✅ Total chunks created: 474
📄 Total pages: 97

--- Chunk 1 Content ---
AWS Solution Architect
(UNIT-2:Compute & Storage)
Dr. Shagun Sharma
Assistant Professor
School of Computing Science and Engineering
shagunsharma@vitbhopal.ac.in
1

--- Chunk 1 Metadata ---
{'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2026-03-15T04:19:45+00:00', 'moddate': '2026-03-15T04:19:46+00:00', 'source': 'data/aws_unit_2.pdf', 'total_pages': 97, 'page': 0, 'page_label': '1'}


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embeddings,
    persist_directory = "./chroma_db"
)

print("vector store created successfully")
print(f"total vectors stored: {vectorstore._collection.count()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10115.04it/s]


vector store created successfully
total vectors stored: 474


In [18]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(
    model_name = "llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)

prompt = PromptTemplate.from_template("""Use the following context to answer the question.
If you don't know the answer, say "I don't know".

Context: {context}

Question: {question}

Answer:
""")

retriever = vectorstore.as_retriever(search_kwargs={"k":4})

chain = (
    {"context": retriever, "question":RunnablePassthrough()}
    | prompt
    | llm
    |StrOutputParser()
)

question = "What is Amazon EC2?"
answer = chain.invoke(question)

print(f"Question: {question}")
print(f"\n Answer: {answer}")

Question: What is Amazon EC2?

 Answer: Amazon EC2 (Elastic Compute Cloud) is a web service provided by the AWS cloud that is secure, resizable, and scalable. It allows users to rent a powerful, virtual computer in the cloud that can be used to run any application.


In [19]:
docs = retriever.invoke(question)

print(f"📄 Source chunks used to answer: '{question}'\n")
for i, doc in enumerate(docs):
    print(f"--- Chunk {i+1} ---")
    print(f"Page: {doc.metadata['page']}")
    print(f"Content preview: {doc.page_content[:200]}")
    print()

📄 Source chunks used to answer: 'What is Amazon EC2?'

--- Chunk 1 ---
Page: 3
Content preview: Amazon EC2 (Elastic Compute Cloud)
• Amazon Elastic Compute Cloud (Amazon EC2) is a web service which
is provided by the AWS cloud which is secure, resizable, and scalable.

--- Chunk 2 ---
Page: 12
Content preview: • It allows its users to choose from various software present to run on their EC2 
machines. 
• This whole service is allocated to AWS Marketplace on the AWS platform.

--- Chunk 3 ---
Page: 2
Content preview: Amazon EC2 (Elastic Compute Cloud)
• Amazon EC2 is like renting a powerful, virtual computer in the cloud
that you can use to run any application.

--- Chunk 4 ---
Page: 77
Content preview: Amazon EBS (Elastic Block Store)
• EC2 is a virtual server in a cloud while EBS is a virtual disk in a cloud.
• Amazon EBS allows you to create storage volumes and attach them to 
the EC2 instances.

